# 🚦 Phase 5: Image Preprocessing Pipeline
### Traffic Sign Recognition using Deep Learning & CNN

---

## 🎯 Objectives for Phase 5:
1. **Uniform Resizing**: Standardize diverse image sizes ($25\times25$ to $200\times200$) into uniform **$32 \times 32 \times 3$** pixel tensors.
2. **Color Space Conversion**: Ensure images read by OpenCV (BGR) are properly mapped to RGB.
3. **Pixel Normalization**: Scale raw pixel intensities from integer $[0, 255]$ to floating-point $[0.0, 1.0]$.
4. **Stratified Train/Test Split**: Split dataset 80% for training and 20% for testing while strictly preserving the class ratios across all 43 classes.
5. **Domain-Safe Augmentation**: Explore realistic transformations (minor rotation $\pm 10^\circ$, shifts, zoom) and understand **why horizontal flips are prohibited** for directional traffic signs.
6. **Dataset Caching**: Save preprocessed NumPy arrays to `dataset/processed_data.npz` so future training notebooks load in under 2 seconds!

> **Beginner Tip:** Run each cell sequentially using `Shift + Enter`.

### 📁 Step 5.1: Import Libraries & Configure Paths

In [ ]:
import os
import sys
import cv2
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Handle project root path whether run from notebooks/ or root
if os.path.basename(os.getcwd()) == "notebooks":
    PROJECT_ROOT = os.path.abspath("..")
else:
    PROJECT_ROOT = os.path.abspath(".")

sys.path.append(PROJECT_ROOT)
from config import DATA_PATH, DATASET_DIR, IMAGE_SIZE, TEST_SPLIT_RATIO, RANDOM_STATE
from label_utils import CLASS_NAMES

print("Project Root :", PROJECT_ROOT)
print("Dataset Path :", DATA_PATH)
print("Target Shape :", (IMAGE_SIZE[0], IMAGE_SIZE[1], 3))

if not os.path.exists(DATA_PATH):
    print("\n⚠️ Dataset folder not found! Please run Phase 2 notebook first.")
else:
    print("✅ Dataset folder verified!")

### 🖼️ Step 5.2: Resizing & BGR-to-RGB Conversion
OpenCV loads images in **BGR** (Blue, Green, Red) order by default, whereas neural networks and Matplotlib expect **RGB** (Red, Green, Blue).
Let's see what happens if we forget to convert BGR to RGB!

In [ ]:
# Pick a sample Stop sign (Class 14)
sample_folder = os.path.join(DATA_PATH, "14")
sample_file = os.path.join(sample_folder, os.listdir(sample_folder)[0])

# 1. Read with OpenCV (returns BGR)
raw_bgr = cv2.imread(sample_file)
original_shape = raw_bgr.shape

# 2. Resize to 32x32
resized_bgr = cv2.resize(raw_bgr, IMAGE_SIZE)

# 3. Convert to RGB
resized_rgb = cv2.cvtColor(resized_bgr, cv2.COLOR_BGR2RGB)

# Compare side-by-side
plt.figure(figsize=(10, 3.5))

plt.subplot(1, 3, 1)
plt.imshow(raw_bgr)  # Shows with swapped blue/red colors
plt.title(f"Raw BGR (No Conversion)\nShape: {original_shape[1]}x{original_shape[0]}", fontsize=9.5, color='red')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(resized_bgr)
plt.title("Resized to 32x32 (Still BGR)", fontsize=9.5, color='orange')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(resized_rgb)
plt.title("Resized + RGB Converted\nShape: 32x32x3 (Correct!)", fontsize=9.5, color='green')
plt.axis('off')

plt.tight_layout()
plt.show()

print("💡 Viva Point: OpenCV reads images as BGR. We must convert to RGB so red stop signs stay red instead of blue!")

### 🔢 Step 5.3: Pixel Normalization ($X / 255.0$)
Raw pixel values range from `0` to `255` (integers).
Dividing by `255.0` transforms all values into the range `[0.0, 1.0]` (floating point numbers).
- **Why normalize?** Neural network weights are initialized to small values. Unnormalized large inputs ($>100$) cause gradients to explode, leading to unstable training.

In [ ]:
raw_pixel_sample = resized_rgb[:2, :2, 0]  # Top-left 2x2 red channel pixels
normalized_sample = raw_pixel_sample / 255.0

print("--- Before Normalization (Integers 0-255) ---")
print(raw_pixel_sample)
print(f"Data type: {raw_pixel_sample.dtype}, Min: {resized_rgb.min()}, Max: {resized_rgb.max()}")

print("\n--- After Normalization (Floats 0.0 - 1.0) ---")
print(normalized_sample)
print(f"Data type: {normalized_sample.dtype}, Min: {(resized_rgb/255.0).min():.3f}, Max: {(resized_rgb/255.0).max():.3f}")

### 🔄 Step 5.4: Process the Entire GTSRB Dataset
Now we iterate across all 43 folders, resize each image to $32 \times 32$, convert to RGB, and build our NumPy matrices $X$ and $y$.

In [ ]:
images = []
labels = []
start_time = time.time()

print("Loading and preprocessing all 43 classes...")
for cid in range(43):
    folder = os.path.join(DATA_PATH, str(cid))
    if not os.path.exists(folder):
        continue
    
    files = [f for f in os.listdir(folder) if f.lower().endswith(('.ppm', '.png', '.jpg', '.jpeg'))]
    for fname in files:
        fpath = os.path.join(folder, fname)
        img = cv2.imread(fpath)
        if img is not None:
            img_resized = cv2.resize(img, IMAGE_SIZE)
            img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
            images.append(img_rgb)
            labels.append(cid)

# Convert to standardized NumPy arrays
X = np.array(images, dtype='float32') / 255.0
y = np.array(labels, dtype='int32')

elapsed = time.time() - start_time
print(f"\nDone in {elapsed:.1f} seconds!")
print(f"Total Images Loaded (X) : {X.shape} (Count, Height, Width, Channels)")
print(f"Total Labels Loaded (y) : {y.shape}")
print(f"Memory usage of X       : {X.nbytes / (1024 * 1024):.1f} MB")

### ✂️ Step 5.5: Stratified Train / Test Split (80% Train, 20% Test)
Because of the class imbalance we discovered in Phase 4, we pass `stratify=y` to `train_test_split`.
This guarantees that both the training set and testing set have the exact same percentage of every single sign category!

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SPLIT_RATIO,
    random_state=RANDOM_STATE,
    stratify=y
)

print("=" * 45)
print("📊 DATASET SPLIT RESULTS")
print("=" * 45)
print(f"Training set samples (X_train) : {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Testing set samples  (X_test)  : {len(X_test):,} ({len(X_test)/len(X)*100:.1f}%)")
print(f"Input shape per image          : {X_train.shape[1:]}")
print("=" * 45)

# Verify stratification: Compare class proportions between train and test
train_prop_0 = np.mean(y_train == 2)  # Proportion of Speed Limit 50 in train
test_prop_0 = np.mean(y_test == 2)    # Proportion of Speed Limit 50 in test
print(f"Class 2 ratio in Train: {train_prop_0*100:.2f}%")
print(f"Class 2 ratio in Test : {test_prop_0*100:.2f}%")
print("✅ Stratification verified: Class distributions are perfectly aligned!")

### ⚠️ Step 5.6: Data Augmentation & The Horizontal Flip Rule
Data augmentation artificially introduces variations to make neural networks more robust against overfitting.

> **CRITICAL VIVA RULE:**
> In standard computer vision (like cat vs. dog classification), **horizontal flipping** is common.
> **HOWEVER, in traffic sign recognition, horizontal flips are FORBIDDEN** on directional signs!
> - Flipping "Turn Right" (Class 33) makes it "Turn Left" (Class 34)!
> - Flipping "Keep Right" (Class 38) makes it "Keep Left" (Class 39)!
> - Flipping speed limit numbers like "50" reverses the digits into illegible shapes.

Let's visualize this danger clearly:

In [ ]:
# Find a 'Turn Right Ahead' sign (Class 33)
turn_right_idx = np.where(y_train == 33)[0][0]
original_sign = X_train[turn_right_idx]
flipped_sign = np.fliplr(original_sign)

plt.figure(figsize=(7, 3))

plt.subplot(1, 2, 1)
plt.imshow(original_sign)
plt.title(f"Ground Truth: Class 33\n'{CLASS_NAMES[33]}'", color='green', fontsize=10)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(flipped_sign)
plt.title("Horizontally Flipped:\nPhysically becomes 'Turn Left'!", color='red', fontsize=10)
plt.axis('off')

plt.tight_layout()
plt.show()

print("💡 Safe augmentations: Small rotation (±10°), small zoom (±10%), brightness adjustments.")
print("🚫 Unsafe augmentations: Horizontal and vertical flipping.")

### 💾 Step 5.7: Cache Preprocessed Arrays for Fast Training
We save our normalized `X_train`, `X_test`, `y_train`, and `y_test` into `dataset/processed_data.npz`.
In Phase 6 and 7, we can load everything instantly in 1 second instead of waiting to re-read thousands of individual files from disk.

In [ ]:
cache_path = os.path.join(DATASET_DIR, "processed_data.npz")

print(f"Saving preprocessed data to: {cache_path} ...")
np.savez_compressed(
    cache_path,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)

file_size_mb = os.path.getsize(cache_path) / (1024 * 1024)
print(f"✅ Successfully saved! Compressed cache size: {file_size_mb:.1f} MB")
print("🎉 Phase 5 Complete: Ready for CNN architecture construction in Phase 6!")

## 🏆 Phase 5 Summary & Next Step
In Phase 5, we have:
- [x] Standardized all images to $32 \times 32 \times 3$
- [x] Converted BGR color format to RGB
- [x] Scaled pixel values to $[0.0, 1.0]$
- [x] Partitioned data into 80% train and 20% test using stratified sampling
- [x] Learned why horizontal flipping is forbidden on directional signs
- [x] Cached preprocessed tensors into `dataset/processed_data.npz`

👉 **You are now ready for Phase 6: CNN Architecture Design (`06_phase6_cnn_architecture.ipynb`)!**